# Lab — Enterprise visual quality inspection

Build an auditable five-class quality gate from pixels to learned representations. You will create and profile a multi-source dataset, train a CNN from scratch, reuse two real pretrained encoders, inspect their embedding geometry, inject a deployment shift, analyse failures, and save an enterprise decision artifact.

**Classes:** Normal · Surface Defect · Structural Defect · Contamination · Unknown / Ambiguous

**Decision boundary:** this is a learning system on synthetic data, not a validated inspection model. The final policy must preserve a human-review route.

![Enterprise vision pipeline](assets/enterprise-vision-pipeline.svg)

## 0. Objectives, success criteria, and experiment contract

By the end of this notebook you should be able to explain:

- what each image tensor dimension and normalization step means;
- how convolution and receptive field create a visual hierarchy;
- why source-aware splitting and duplicate checks precede training;
- what frozen transfer and partial fine-tuning change;
- what nearest neighbours and PCA reveal—and what they do not;
- how clean performance differs from shifted performance; and
- why a model score becomes an enterprise decision only after thresholds, abstention, and monitoring.

The default configuration is deliberately CPU-friendly. Set `CV_FULL_RUN=1` before launching Jupyter for a larger generated dataset and longer training. Both modes execute the same real pipeline; neither mocks model outputs.

In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import os
import platform
import random
import shutil
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
import sklearn
import torch
import torchvision
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter, ImageOps
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import (
    ConvNeXt_Tiny_Weights,
    ResNet18_Weights,
    convnext_tiny,
    resnet18,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

cwd = Path.cwd().resolve()
relative_course = Path("curriculum/beginner/01-modern-computer-vision-foundations")
COURSE_DIR = cwd / relative_course if (cwd / relative_course).exists() else cwd
ARTIFACT_DIR = COURSE_DIR / ".artifacts" / "enterprise_quality_inspection"
DATA_DIR = ARTIFACT_DIR / "dataset"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

FULL_RUN = os.getenv("CV_FULL_RUN", "0") == "1"
DEVICE = torch.device("cpu")  # stable cross-platform teaching baseline

@dataclass(frozen=True)
class Config:
    image_size: int = 96
    pretrained_crop: int = 128
    train_per_class: int = 48 if FULL_RUN else 20
    val_per_class: int = 16 if FULL_RUN else 8
    test_per_class: int = 20 if FULL_RUN else 10
    scratch_epochs: int = 12 if FULL_RUN else 5
    partial_epochs: int = 4 if FULL_RUN else 1
    batch_size: int = 16
    review_threshold: float = 0.70

CFG = Config()
CLASS_NAMES = [
    "Normal",
    "Surface Defect",
    "Structural Defect",
    "Contamination",
    "Unknown / Ambiguous",
]
CLASS_SLUGS = ["normal", "surface_defect", "structural_defect", "contamination", "unknown_ambiguous"]
CLASS_TO_ID = {name: index for index, name in enumerate(CLASS_NAMES)}

versions = pd.Series({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pillow": PIL.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "device": str(DEVICE),
    "full_run": FULL_RUN,
})
display(versions.to_frame("value"))
print("Artifacts:", ARTIFACT_DIR)

## 1. Generate a controlled multi-source dataset

Public industrial benchmarks are valuable, but they may have noncommercial licences or a different label protocol. Here we generate a deterministic dataset that makes the experiment auditable:

- `Factory_A` and `Factory_B` supply training/validation data.
- `Factory_C` is held out until final evaluation.
- known-source backgrounds contain a label-correlated tint, creating a learnable shortcut;
- the held-out source removes that correlation;
- one exact image is deliberately copied from train to validation so the profile must catch it.

This design is a teaching instrument. Synthetic performance is not evidence of real-world readiness.

In [ ]:
def make_component(label_id: int, source: str, sample_seed: int, size: int) -> Image.Image:
    rng = np.random.default_rng(sample_seed)
    yy, xx = np.mgrid[0:size, 0:size]

    source_bias = {"Factory_A": -10, "Factory_B": 8, "Factory_C": 0}[source]
    shortcut = (label_id - 2) * 12 if source != "Factory_C" else 0
    base = np.clip(142 + source_bias + shortcut + 0.10 * (xx - size / 2), 55, 220)
    noise = rng.normal(0, 5, (size, size))
    plate = np.clip(base + noise, 0, 255).astype(np.uint8)
    rgb = np.stack([plate, np.clip(plate + 3, 0, 255), np.clip(plate + 7, 0, 255)], axis=-1).astype(np.uint8)
    image = Image.fromarray(rgb, mode="RGB")
    draw = ImageDraw.Draw(image, "RGBA")

    margin = size // 8
    draw.rounded_rectangle((margin, margin, size - margin, size - margin), radius=10, outline=(35, 45, 55, 220), width=3)
    for x, y in [(margin + 8, margin + 8), (size - margin - 8, margin + 8), (margin + 8, size - margin - 8), (size - margin - 8, size - margin - 8)]:
        draw.ellipse((x - 3, y - 3, x + 3, y + 3), fill=(55, 65, 75, 230))

    if label_id == 1:  # surface scratches
        for _ in range(3):
            x0 = int(rng.integers(margin + 8, size // 2))
            y0 = int(rng.integers(margin + 8, size - margin - 8))
            x1 = int(rng.integers(size // 2, size - margin - 5))
            draw.line((x0, y0, x1, y0 + int(rng.integers(-7, 8))), fill=(235, 235, 225, 235), width=2)
            draw.line((x0, y0 + 2, x1, y0 + int(rng.integers(-7, 8)) + 2), fill=(50, 55, 60, 170), width=1)
    elif label_id == 2:  # structural crack
        points = [(margin + 10, size // 3)]
        for step in range(1, 6):
            points.append((margin + 10 + step * (size - 2 * margin - 20) // 5, size // 3 + int(rng.integers(-11, 12))))
        draw.line(points, fill=(25, 20, 25, 245), width=4)
        draw.line([(x, y + 2) for x, y in points], fill=(225, 205, 190, 130), width=1)
    elif label_id == 3:  # contamination
        for _ in range(7):
            x = int(rng.integers(margin + 8, size - margin - 8))
            y = int(rng.integers(margin + 8, size - margin - 8))
            radius = int(rng.integers(3, 9))
            colour = (95, int(rng.integers(60, 105)), 30, int(rng.integers(100, 190)))
            draw.ellipse((x - radius, y - radius, x + radius, y + radius), fill=colour)
    elif label_id == 4:  # intentionally ambiguous signal
        x0, y0 = margin + 12, int(rng.integers(size // 3, 2 * size // 3))
        draw.line((x0, y0, size - margin - 12, y0 + int(rng.integers(-5, 6))), fill=(70, 65, 70, 110), width=2)
        draw.ellipse((size // 2 - 7, size // 2 - 5, size // 2 + 8, size // 2 + 6), fill=(100, 75, 40, 70))
        draw.rectangle((size // 2 - 11, margin + 2, size // 2 + 11, size - margin - 2), fill=(210, 210, 210, 35))

    if source == "Factory_B":
        image = ImageEnhance.Contrast(image).enhance(1.08)
    elif source == "Factory_C":
        image = ImageEnhance.Color(image).enhance(0.82)
    return image


def generate_dataset() -> pd.DataFrame:
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    DATA_DIR.mkdir(parents=True)
    rows = []
    split_spec = {
        "train": (["Factory_A", "Factory_B"], CFG.train_per_class),
        "val": (["Factory_A", "Factory_B"], CFG.val_per_class),
        "test": (["Factory_C"], CFG.test_per_class),
    }
    running_id = 0
    for split, (sources, per_class) in split_spec.items():
        for label_id, (label, slug) in enumerate(zip(CLASS_NAMES, CLASS_SLUGS)):
            for index in range(per_class):
                source = sources[index % len(sources)]
                sample_id = f"{split}_{running_id:04d}"
                path = DATA_DIR / split / slug / f"{sample_id}.png"
                path.parent.mkdir(parents=True, exist_ok=True)
                seed = SEED * 10_000 + running_id
                make_component(label_id, source, seed, CFG.image_size).save(path)
                rows.append({
                    "sample_id": sample_id,
                    "path": str(path),
                    "split": split,
                    "source": source,
                    "label": label,
                    "label_id": label_id,
                    "duplicate_of": None,
                })
                running_id += 1

    frame = pd.DataFrame(rows)
    original = frame.query("split == 'train' and label == 'Normal'").iloc[0]
    duplicate_path = DATA_DIR / "val" / "normal" / "val_deliberate_duplicate.png"
    duplicate_path.write_bytes(Path(original.path).read_bytes())
    duplicate = {
        "sample_id": "val_deliberate_duplicate",
        "path": str(duplicate_path),
        "split": "val",
        "source": "Factory_B",
        "label": "Normal",
        "label_id": 0,
        "duplicate_of": original.sample_id,
    }
    return pd.concat([frame, pd.DataFrame([duplicate])], ignore_index=True)


raw_df = generate_dataset()
print(f"Generated {len(raw_df):,} records under {DATA_DIR}")
display(pd.crosstab([raw_df["split"], raw_df["source"]], raw_df["label"]))

## 2. Profile before modelling

A file count is not a data profile. We verify readability, dimensions, channels, numerical extrema, and hashes. The exact duplicate check uses file bytes here because the deliberate copy is byte-identical. Real pipelines should also compare decoded pixels and use perceptual or embedding similarity for resized/re-encoded near duplicates.

In [ ]:
def inspect_file(path_value: str) -> pd.Series:
    path = Path(path_value)
    payload = path.read_bytes()
    with Image.open(path) as image:
        array = np.asarray(image.convert("RGB"))
    return pd.Series({
        "sha256": hashlib.sha256(payload).hexdigest(),
        "width": array.shape[1],
        "height": array.shape[0],
        "channels": array.shape[2],
        "pixel_min": int(array.min()),
        "pixel_max": int(array.max()),
    })


profile = raw_df["path"].apply(inspect_file)
profiled_df = pd.concat([raw_df, profile], axis=1)
duplicate_rows = profiled_df[profiled_df.duplicated("sha256", keep=False)].sort_values("sha256")
display(profiled_df.groupby("split")[["width", "height", "channels", "pixel_min", "pixel_max"]].agg(["min", "max"]))
display(duplicate_rows[["sample_id", "split", "source", "label", "duplicate_of", "sha256"]])

cross_split_duplicate_hashes = set(
    duplicate_rows.groupby("sha256")["split"].nunique().loc[lambda values: values > 1].index
)
remove_mask = (profiled_df["split"] != "train") & profiled_df["sha256"].isin(cross_split_duplicate_hashes)
clean_df = profiled_df.loc[~remove_mask].reset_index(drop=True)
assert not clean_df.groupby("sha256")["split"].nunique().gt(1).any()
assert set(clean_df.query("split == 'test'")["source"]) == {"Factory_C"}
print(f"Removed {int(remove_mask.sum())} cross-split duplicate; {len(clean_df)} clean records remain.")

In [ ]:
fig, axes = plt.subplots(len(CLASS_NAMES), 4, figsize=(10, 12))
for row, class_name in enumerate(CLASS_NAMES):
    examples = clean_df.query("label == @class_name").sample(4, random_state=SEED + row)
    for axis, (_, record) in zip(axes[row], examples.iterrows()):
        axis.imshow(Image.open(record.path))
        axis.set_title(f"{class_name}\n{record.source} · {record.split}", fontsize=8)
        axis.axis("off")
fig.suptitle("Profiled samples: inspect labels and sources before training", y=1.01)
plt.tight_layout()

## 3. Image tensors: inspect every boundary

An image pipeline is a sequence of explicit contracts, not a single `load()` call:

`PIL RGB image → NumPy HWC uint8 → HWC float [0,1] → PyTorch CHW → NCHW batch → normalized tensor`

The next cell prints shape, dtype, range, channel statistics, and one pixel at every boundary. It then makes four common mistakes visible: RGB/BGR reversal, passing HWC where NCHW is expected, normalizing twice, and forcing a rectangular image into a square. These bugs can leave the program running while silently changing the problem the model sees.

In [ ]:
sample_path = clean_df.query("label == 'Structural Defect'").iloc[0].path
pil_image = Image.open(sample_path).convert("RGB")
uint8_hwc = np.asarray(pil_image)
float_hwc = uint8_hwc.astype(np.float32) / 255.0
float_chw = torch.from_numpy(float_hwc).permute(2, 0, 1).contiguous()
batch_nchw = float_chw.unsqueeze(0)
mean = torch.tensor([0.5, 0.5, 0.5])[:, None, None]
std = torch.tensor([0.25, 0.25, 0.25])[:, None, None]
normalized_chw = (float_chw - mean) / std

def tensor_contract(name, value, channel_axis):
    array = value.detach().cpu().numpy() if torch.is_tensor(value) else np.asarray(value)
    reduce_axes = tuple(axis for axis in range(array.ndim) if axis != channel_axis)
    channel_mean = np.mean(array, axis=reduce_axes)
    channel_std = np.std(array, axis=reduce_axes)
    if array.ndim == 3 and channel_axis == 2:
        pixel = array[20, 20]
    elif array.ndim == 3:
        pixel = array[:, 20, 20]
    else:
        pixel = array[0, :, 20, 20]
    return {
        "boundary": name,
        "shape": str(tuple(array.shape)),
        "dtype": str(array.dtype),
        "min": float(array.min()),
        "max": float(array.max()),
        "channel_mean": np.round(channel_mean, 3).tolist(),
        "channel_std": np.round(channel_std, 3).tolist(),
        "pixel[20,20]": np.round(pixel, 3).tolist(),
    }

contract_rows = [
    tensor_contract("NumPy HWC uint8", uint8_hwc, 2),
    tensor_contract("NumPy HWC float", float_hwc, 2),
    tensor_contract("PyTorch CHW", float_chw, 0),
    tensor_contract("PyTorch NCHW", batch_nchw, 1),
    tensor_contract("normalized CHW", normalized_chw, 0),
]
display(pd.DataFrame(contract_rows))

bgr_hwc = float_hwc[..., ::-1].copy()
wrong_nchw = torch.from_numpy(float_hwc).unsqueeze(0)  # [N,H,W,C], not [N,C,H,W]
double_normalized = (normalized_chw - mean) / std
rectangular = pil_image.crop((0, CFG.image_size // 6, CFG.image_size, 5 * CFG.image_size // 6))
forced_square = rectangular.resize((CFG.image_size, CFG.image_size), Image.Resampling.BILINEAR)
letterboxed = ImageOps.pad(rectangular, (CFG.image_size, CFG.image_size), method=Image.Resampling.BILINEAR, color=(20, 20, 20))

try:
    assert wrong_nchw.shape[1] == 3, "channel dimension must be index 1 for NCHW"
except AssertionError as error:
    print("Caught layout violation:", error, tuple(wrong_nchw.shape))
print("Single normalization range:", tuple(round(float(x), 2) for x in (normalized_chw.min(), normalized_chw.max())))
print("Double normalization range:", tuple(round(float(x), 2) for x in (double_normalized.min(), double_normalized.max())))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
views = [
    (pil_image, "PIL RGB"),
    (float_hwc[..., 0], "R channel"),
    (float_hwc[..., 1], "G channel"),
    (float_hwc[..., 2], "B channel"),
    (bgr_hwc, "Wrong: BGR as RGB"),
    (rectangular, f"Original aspect {rectangular.size}"),
    (forced_square, "Wrong: forced square"),
    (letterboxed, "Aspect preserved + pad"),
]
for axis, (view, title) in zip(axes.flat, views):
    axis.imshow(view, cmap="gray" if np.asarray(view).ndim == 2 else None)
    axis.set_title(title, fontsize=9)
    axis.axis("off")
plt.tight_layout()


### Convolution: manual loops, a framework primitive, and learned filters

Deep-learning libraries implement cross-correlation under the name convolution. For a single channel,

$$
y[i,j] = \sum_{u=0}^{k_h-1}\sum_{v=0}^{k_w-1} x[i+u,j+v]K[u,v].
$$

We first calculate this with transparent loops, then verify it against `torch.nn.functional.conv2d`. A later cell inspects filters learned by the scratch CNN.

In [ ]:
def manual_xcorr2d(image_2d: torch.Tensor, kernel_2d: torch.Tensor, padding: int = 0) -> torch.Tensor:
    padded = F.pad(image_2d, (padding, padding, padding, padding))
    height = padded.shape[0] - kernel_2d.shape[0] + 1
    width = padded.shape[1] - kernel_2d.shape[1] + 1
    output = torch.empty((height, width), dtype=image_2d.dtype)
    for row in range(height):
        for column in range(width):
            patch = padded[row:row + kernel_2d.shape[0], column:column + kernel_2d.shape[1]]
            output[row, column] = (patch * kernel_2d).sum()
    return output

gray_patch = batch_nchw.mean(dim=1, keepdim=True)[0, 0, 24:72, 24:72]
sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])
manual_response = manual_xcorr2d(gray_patch, sobel_x, padding=1)
framework_response = F.conv2d(gray_patch[None, None], sobel_x[None, None], padding=1)[0, 0]
maximum_error = (manual_response - framework_response).abs().max().item()
assert torch.allclose(manual_response, framework_response, atol=1e-6)
print(f"manual vs F.conv2d maximum absolute error: {maximum_error:.2e}")

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for axis, view, title in [
    (axes[0], gray_patch, "grayscale crop"),
    (axes[1], manual_response, "manual Sobel response"),
    (axes[2], framework_response, "F.conv2d response"),
]:
    axis.imshow(view.detach().numpy(), cmap="coolwarm" if "response" in title else "gray")
    axis.set_title(title, fontsize=9)
    axis.axis("off")
plt.tight_layout()


### Receptive-field arithmetic before training

The theoretical receptive field is the largest input region that can affect an activation. For layer $l$ with kernel $k_l$, dilation $d_l$, stride $s_l$, and previous jump $j_{l-1}$,

$$
r_l=r_{l-1}+(k_l-1)d_lj_{l-1}, \qquad j_l=j_{l-1}s_l.
$$

The arithmetic below predicts context size. After training, we compare it with the pixels that produce non-zero gradients for a chosen activation—the empirical support for that input and parameter state.

In [ ]:
layers = [
    {"name": "conv1", "kernel": 3, "stride": 1, "dilation": 1},
    {"name": "pool1", "kernel": 2, "stride": 2, "dilation": 1},
    {"name": "conv2", "kernel": 3, "stride": 1, "dilation": 1},
    {"name": "pool2", "kernel": 2, "stride": 2, "dilation": 1},
    {"name": "conv3", "kernel": 3, "stride": 1, "dilation": 1},
]
receptive_field, jump = 1, 1
rf_rows = []
for layer in layers:
    receptive_field += (layer["kernel"] - 1) * layer["dilation"] * jump
    jump *= layer["stride"]
    rf_rows.append({**layer, "effective_jump": jump, "receptive_field": receptive_field})
display(pd.DataFrame(rf_rows))

## 4. Leakage-safe datasets and preprocessing

The scratch model uses moderate, label-preserving augmentation on training data only. Evaluation is deterministic. Pretrained encoders later use the preprocessing bundled with their official weight objects, preventing an easy but damaging contract mismatch.

In [ ]:
scratch_train_transform = transforms.Compose([
    transforms.Resize((CFG.image_size, CFG.image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.12, contrast=0.12),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.25] * 3),
])
scratch_eval_transform = transforms.Compose([
    transforms.Resize((CFG.image_size, CFG.image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.25] * 3),
])

class InspectionDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform, shift_fn=None):
        self.frame = frame.reset_index(drop=True).copy()
        self.transform = transform
        self.shift_fn = shift_fn

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = Image.open(row.path).convert("RGB")
        if self.shift_fn is not None:
            image = self.shift_fn(image)
        return self.transform(image), int(row.label_id), index


splits = {name: clean_df.query("split == @name").reset_index(drop=True) for name in ["train", "val", "test"]}
train_ds = InspectionDataset(splits["train"], scratch_train_transform)
val_ds = InspectionDataset(splits["val"], scratch_eval_transform)
test_ds = InspectionDataset(splits["test"], scratch_eval_transform)
train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False)

images, labels, indices = next(iter(train_loader))
print("Batch contract:", tuple(images.shape), images.dtype, "labels:", tuple(labels.shape))

## 5. Baseline — train a compact CNN from scratch

A scratch model is essential evidence. It tells us whether the task is learnable with local patterns and whether pretrained features add value relative to their cost. The architecture is intentionally small: three convolution blocks, global average pooling, and a linear head.

In [ ]:
class ScratchCNN(nn.Module):
    def __init__(self, class_count: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 24, 3, padding=1), nn.BatchNorm2d(24), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(24, 48, 3, padding=1), nn.BatchNorm2d(48), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(48, 96, 3, padding=1), nn.BatchNorm2d(96), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(96, class_count)

    def forward(self, inputs):
        return self.classifier(self.features(inputs).flatten(1))


def train_model(model, train_data, val_data, epochs, learning_rate, weight_decay=1e-4):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    criterion = nn.CrossEntropyLoss()
    history, best_state, best_f1 = [], copy.deepcopy(model.state_dict()), -1.0
    started = time.perf_counter()
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for batch, targets, _ in train_data:
            batch, targets = batch.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch), targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(targets)
        val_result = predict_torch(model, val_data)
        val_f1 = f1_score(val_result["y_true"], val_result["y_pred"], average="macro", zero_division=0)
        history.append({"epoch": epoch, "train_loss": train_loss / len(train_data.dataset), "val_macro_f1": val_f1})
        if val_f1 > best_f1:
            best_f1, best_state = val_f1, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), time.perf_counter() - started


@torch.inference_mode()
def predict_torch(model, loader):
    model.eval()
    labels, probabilities, row_indices = [], [], []
    started = time.perf_counter()
    for batch, targets, indices in loader:
        logits = model(batch.to(DEVICE))
        probabilities.append(logits.softmax(dim=1).cpu().numpy())
        labels.append(targets.numpy())
        row_indices.append(indices.numpy())
    probs = np.concatenate(probabilities)
    return {
        "y_true": np.concatenate(labels),
        "y_pred": probs.argmax(axis=1),
        "probs": probs,
        "row_indices": np.concatenate(row_indices),
        "seconds": time.perf_counter() - started,
    }


def quality_metrics(y_true, probs, threshold=CFG.review_threshold):
    y_pred = probs.argmax(axis=1)
    is_defect = np.asarray(y_true) != 0
    predicted_defect = y_pred != 0
    review = (probs.max(axis=1) < threshold) | (y_pred == CLASS_TO_ID["Unknown / Ambiguous"])
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "defect_recall": recall_score(is_defect, predicted_defect, zero_division=0),
        "normal_recall": recall_score(~is_defect, ~predicted_defect, zero_division=0),
        "review_rate": review.mean(),
    }


scratch_model = ScratchCNN(len(CLASS_NAMES))
scratch_parameters = sum(parameter.numel() for parameter in scratch_model.parameters())
scratch_model, scratch_history, scratch_train_seconds = train_model(
    scratch_model, train_loader, val_loader, CFG.scratch_epochs, learning_rate=2e-3
)
display(scratch_history)
print(f"Scratch parameters: {scratch_parameters:,}; train time: {scratch_train_seconds:.2f}s")

In [ ]:
scratch_clean = predict_torch(scratch_model, test_loader)
display(pd.Series(quality_metrics(scratch_clean["y_true"], scratch_clean["probs"]), name="scratch_clean").to_frame())
print(classification_report(scratch_clean["y_true"], scratch_clean["y_pred"], target_names=CLASS_NAMES, zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    scratch_clean["y_true"], scratch_clean["y_pred"], display_labels=CLASS_NAMES, xticks_rotation=35, cmap="Blues"
)
plt.title("Scratch CNN · held-out Factory_C")
plt.tight_layout()

### Learned filters, feature maps, and empirical receptive fields

The first plots show what the scratch CNN actually learned. The table derives the theoretical receptive field directly from its modules; the gradient masks show the active input support for one central feature. The empirical support can be smaller because ReLUs and learned weights block some paths.

In [ ]:
scratch_model.eval()
example_record = splits["test"].query("label == 'Structural Defect'").iloc[0]
example_tensor = scratch_eval_transform(Image.open(example_record.path).convert("RGB")).unsqueeze(0).to(DEVICE)

first_filters = scratch_model.features[0].weight.detach().cpu()
with torch.inference_mode():
    first_features = scratch_model.features[:3](example_tensor).cpu()[0]

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for index in range(8):
    kernel = first_filters[index].permute(1, 2, 0).numpy()
    kernel = (kernel - kernel.min()) / (kernel.max() - kernel.min() + 1e-8)
    axes[0, index].imshow(kernel)
    axes[0, index].set_title(f"filter {index}", fontsize=8)
    axes[1, index].imshow(first_features[index], cmap="viridis")
    axes[1, index].set_title(f"map {index}", fontsize=8)
    axes[0, index].axis("off")
    axes[1, index].axis("off")
axes[0, 0].set_ylabel("learned kernel")
axes[1, 0].set_ylabel("feature map")
plt.tight_layout()

rf_rows = []
receptive_field, jump = 1, 1
for module_index, module in enumerate(scratch_model.features):
    if isinstance(module, (nn.Conv2d, nn.MaxPool2d)):
        kernel = module.kernel_size[0] if isinstance(module.kernel_size, tuple) else module.kernel_size
        stride = module.stride[0] if isinstance(module.stride, tuple) else module.stride
        dilation_value = getattr(module, "dilation", 1)
        dilation = dilation_value[0] if isinstance(dilation_value, tuple) else dilation_value
        receptive_field += (kernel - 1) * dilation * jump
        jump *= stride
        rf_rows.append({
            "module_index": module_index,
            "layer": type(module).__name__,
            "kernel": kernel,
            "stride": stride,
            "jump": jump,
            "theoretical_rf": receptive_field,
        })
rf_table = pd.DataFrame(rf_rows)
display(rf_table)

stage_indices = [0, 4, 8]
fig, axes = plt.subplots(1, len(stage_indices), figsize=(10, 3))
empirical_rows = []
for axis, stage_index in zip(axes, stage_indices):
    probe_input = example_tensor.detach().clone().requires_grad_(True)
    stage_output = scratch_model.features[:stage_index + 1](probe_input)
    center = stage_output[0, stage_output.shape[1] // 2, stage_output.shape[2] // 2, stage_output.shape[3] // 2]
    gradient = torch.autograd.grad(center, probe_input)[0].abs().sum(dim=1)[0].detach().cpu()
    support = gradient > gradient.max() * 1e-6
    coordinates = support.nonzero()
    empirical_height = int(coordinates[:, 0].max() - coordinates[:, 0].min() + 1) if len(coordinates) else 0
    empirical_width = int(coordinates[:, 1].max() - coordinates[:, 1].min() + 1) if len(coordinates) else 0
    theoretical = int(rf_table.query("module_index == @stage_index").iloc[0].theoretical_rf)
    empirical_rows.append({"conv_module": stage_index, "theoretical_rf": theoretical, "empirical_support": f"{empirical_height}×{empirical_width}"})
    axis.imshow(gradient, cmap="magma")
    axis.set_title(f"conv {stage_index}\ntheory {theoretical}×{theoretical}; active {empirical_height}×{empirical_width}", fontsize=8)
    axis.axis("off")
display(pd.DataFrame(empirical_rows))
plt.tight_layout()


## 6. Real pretrained encoders: ResNet-18 and ConvNeXt-Tiny

The next experiment downloads official torchvision weights, freezes every encoder parameter, extracts one vector per image, and fits a multinomial logistic-regression probe. The probe is deliberately simple: representation quality must do most of the work.

![Transfer learning options](assets/transfer-learning.svg)

In [ ]:
def build_frozen_encoder(name: str):
    if name == "resnet18":
        weights = ResNet18_Weights.DEFAULT
        model = resnet18(weights=weights)
        dimension = model.fc.in_features
        model.fc = nn.Identity()
    elif name == "convnext_tiny":
        weights = ConvNeXt_Tiny_Weights.DEFAULT
        model = convnext_tiny(weights=weights)
        dimension = model.classifier[-1].in_features
        model.classifier[-1] = nn.Identity()
    else:
        raise ValueError(name)
    for parameter in model.parameters():
        parameter.requires_grad = False
    model.eval().to(DEVICE)
    return model, weights, dimension


@torch.inference_mode()
def extract_features(model, weights, frame, shift_fn=None):
    transform = weights.transforms(crop_size=CFG.pretrained_crop, resize_size=CFG.pretrained_crop + 16)
    loader = DataLoader(
        InspectionDataset(frame, transform, shift_fn=shift_fn),
        batch_size=max(4, CFG.batch_size // 2),
        shuffle=False,
    )
    features, labels = [], []
    started = time.perf_counter()
    for batch, targets, _ in loader:
        features.append(model(batch.to(DEVICE)).cpu().numpy())
        labels.append(targets.numpy())
    return np.concatenate(features), np.concatenate(labels), time.perf_counter() - started


all_ordered = pd.concat([splits["train"], splits["val"], splits["test"]], ignore_index=True)
boundaries = {
    "train": slice(0, len(splits["train"])),
    "val": slice(len(splits["train"]), len(splits["train"]) + len(splits["val"])),
    "test": slice(len(splits["train"]) + len(splits["val"]), len(all_ordered)),
}
encoders, feature_store, probes, extraction_seconds = {}, {}, {}, {}
for encoder_name in ["resnet18", "convnext_tiny"]:
    model, weights, dimension = build_frozen_encoder(encoder_name)
    features, labels, elapsed = extract_features(model, weights, all_ordered)
    probe = LogisticRegression(max_iter=600, class_weight="balanced", random_state=SEED)
    probe.fit(features[boundaries["train"]], labels[boundaries["train"]])
    encoders[encoder_name] = (model, weights, dimension)
    feature_store[encoder_name] = features
    probes[encoder_name] = probe
    extraction_seconds[encoder_name] = elapsed
    print(f"{encoder_name}: {dimension}-D embeddings for {len(features)} images in {elapsed:.2f}s")

In [ ]:
frozen_clean = {}
for encoder_name, probe in probes.items():
    test_slice = boundaries["test"]
    probabilities = probe.predict_proba(feature_store[encoder_name][test_slice])
    frozen_clean[encoder_name] = {
        "y_true": splits["test"]["label_id"].to_numpy(),
        "y_pred": probabilities.argmax(axis=1),
        "probs": probabilities,
    }
    display(pd.Series(quality_metrics(frozen_clean[encoder_name]["y_true"], probabilities), name=encoder_name).to_frame())

## 7. Inspect representation geometry

A useful classifier score can still hide a brittle embedding space. We inspect `ResNet-18` nearest neighbours and a two-dimensional PCA projection. Neighbours should share defect evidence rather than only background. PCA is fitted on training embeddings and then applied to validation/test embeddings, avoiding leakage.

In [ ]:
encoder_name = "resnet18"
train_features = normalize(feature_store[encoder_name][boundaries["train"]])
test_features = normalize(feature_store[encoder_name][boundaries["test"]])
query_indices = [0, len(splits["test"]) // 2, len(splits["test"]) - 1]
fig, axes = plt.subplots(len(query_indices), 4, figsize=(11, 8))
for row, query_index in enumerate(query_indices):
    similarities = train_features @ test_features[query_index]
    neighbour_indices = similarities.argsort()[-3:][::-1]
    query = splits["test"].iloc[query_index]
    axes[row, 0].imshow(Image.open(query.path))
    axes[row, 0].set_title(f"Query\n{query.label}", fontsize=8)
    for column, neighbour_index in enumerate(neighbour_indices, start=1):
        neighbour = splits["train"].iloc[neighbour_index]
        axes[row, column].imshow(Image.open(neighbour.path))
        axes[row, column].set_title(f"{neighbour.label}\nsim={similarities[neighbour_index]:.2f}", fontsize=8)
    for axis in axes[row]:
        axis.axis("off")
fig.suptitle("ResNet-18 embedding nearest neighbours", y=1.01)
plt.tight_layout()

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
train_2d = pca.fit_transform(feature_store[encoder_name][boundaries["train"]])
test_2d = pca.transform(feature_store[encoder_name][boundaries["test"]])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True, sharey=True)
for axis, points, frame, title in [
    (axes[0], train_2d, splits["train"], "Known-source train"),
    (axes[1], test_2d, splits["test"], "Held-out Factory_C"),
]:
    for label_id, class_name in enumerate(CLASS_NAMES):
        mask = frame["label_id"].to_numpy() == label_id
        axis.scatter(points[mask, 0], points[mask, 1], s=24, alpha=0.75, label=class_name)
    axis.set_title(title)
    axis.set_xlabel("PC1")
    axis.grid(alpha=0.2)
axes[0].set_ylabel("PC2")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
fig.suptitle(f"PCA fitted on training embeddings · explained variance {pca.explained_variance_ratio_.sum():.1%}")
plt.tight_layout()

### Embedding-space dataset intelligence

Nearest neighbours are not just a search demo. They can surface exact/near duplicates, label disagreement, out-of-distribution candidates, uncertain hard examples, and source-dominated clusters. These flags are review queues—not automatic proof that a label is wrong or a sample is OOD.

In [ ]:
all_features = normalize(feature_store[encoder_name])
train_count = len(splits["train"])
similarity_to_all = all_features @ all_features.T
np.fill_diagonal(similarity_to_all, -np.inf)
nearest_index = similarity_to_all.argmax(axis=1)
nearest_similarity = similarity_to_all[np.arange(len(all_features)), nearest_index]

similarity_to_train = all_features @ all_features[:train_count].T
similarity_to_train[np.arange(train_count), np.arange(train_count)] = -np.inf
max_train_similarity = similarity_to_train.max(axis=1)

neighbour_count = 5
nearest_five = np.argpartition(similarity_to_all, -neighbour_count, axis=1)[:, -neighbour_count:]
labels_all = all_ordered["label_id"].to_numpy()
sources_all = all_ordered["source"].to_numpy()
label_agreement = np.mean(labels_all[nearest_five] == labels_all[:, None], axis=1)
source_agreement = np.mean(sources_all[nearest_five] == sources_all[:, None], axis=1)

probe_probs_all = probes[encoder_name].predict_proba(feature_store[encoder_name])
sorted_probs = np.sort(probe_probs_all, axis=1)
probability_margin = sorted_probs[:, -1] - sorted_probs[:, -2]
nontrain = np.arange(len(all_features)) >= train_count
ood_threshold = np.quantile(max_train_similarity[nontrain], 0.10)
hard_threshold = np.quantile(probability_margin, 0.10)
near_duplicate_threshold = np.quantile(nearest_similarity, 0.99)

triage = all_ordered[["sample_id", "split", "source", "label", "path", "sha256"]].copy()
triage["nearest_id"] = all_ordered.iloc[nearest_index]["sample_id"].to_numpy()
triage["nearest_label"] = all_ordered.iloc[nearest_index]["label"].to_numpy()
triage["nearest_similarity"] = nearest_similarity
triage["train_similarity"] = max_train_similarity
triage["probability_margin"] = probability_margin
triage["label_agreement@5"] = label_agreement
triage["source_agreement@5"] = source_agreement
triage["exact_duplicate"] = triage.duplicated("sha256", keep=False)
triage["near_duplicate_candidate"] = nearest_similarity >= near_duplicate_threshold
triage["label_disagreement"] = triage["label"] != triage["nearest_label"]
triage["ood_candidate"] = nontrain & (max_train_similarity <= ood_threshold)
triage["hard_example"] = probability_margin <= hard_threshold
triage["source_cluster_candidate"] = source_agreement >= 0.80
flag_columns = ["exact_duplicate", "near_duplicate_candidate", "label_disagreement", "ood_candidate", "hard_example", "source_cluster_candidate"]
triage["flag_count"] = triage[flag_columns].sum(axis=1)
display(triage[flag_columns].sum().rename("flagged samples").to_frame())
display(triage.sort_values(["flag_count", "probability_margin"], ascending=[False, True]).head(10).drop(columns=["path", "sha256"]))

review_rows = triage.sort_values(["flag_count", "probability_margin"], ascending=[False, True]).head(4)
fig, axes = plt.subplots(len(review_rows), 2, figsize=(6, 2.6 * len(review_rows)), squeeze=False)
for row_number, (row_index, row) in enumerate(review_rows.iterrows()):
    neighbour_row = all_ordered.iloc[nearest_index[row_index]]
    axes[row_number, 0].imshow(Image.open(row.path))
    axes[row_number, 1].imshow(Image.open(neighbour_row.path))
    axes[row_number, 0].set_title(f"review: {row.label}\nflags={int(row.flag_count)}", fontsize=8)
    axes[row_number, 1].set_title(f"nearest: {neighbour_row.label}\nsim={row.nearest_similarity:.3f}", fontsize=8)
    axes[row_number, 0].axis("off")
    axes[row_number, 1].axis("off")
plt.tight_layout()


## 8. Partial fine-tuning

Frozen transfer asks whether the existing representation is already sufficient. Partial fine-tuning adapts the final residual stage and classification head using a small learning rate. This increases flexibility and risk: a small dataset can overwrite useful general features or reinforce its shortcuts.

In [ ]:
resnet_weights = ResNet18_Weights.DEFAULT
resnet_transform = resnet_weights.transforms(crop_size=CFG.pretrained_crop, resize_size=CFG.pretrained_crop + 16)
transfer_train_loader = DataLoader(
    InspectionDataset(splits["train"], resnet_transform),
    batch_size=max(4, CFG.batch_size // 2),
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
transfer_val_loader = DataLoader(InspectionDataset(splits["val"], resnet_transform), batch_size=8, shuffle=False)
transfer_test_loader = DataLoader(InspectionDataset(splits["test"], resnet_transform), batch_size=8, shuffle=False)

partial_model = resnet18(weights=resnet_weights)
partial_model.fc = nn.Linear(partial_model.fc.in_features, len(CLASS_NAMES))
for parameter in partial_model.parameters():
    parameter.requires_grad = False
for parameter in partial_model.layer4.parameters():
    parameter.requires_grad = True
for parameter in partial_model.fc.parameters():
    parameter.requires_grad = True
partial_trainable = sum(parameter.numel() for parameter in partial_model.parameters() if parameter.requires_grad)
partial_model, partial_history, partial_train_seconds = train_model(
    partial_model, transfer_train_loader, transfer_val_loader, CFG.partial_epochs, learning_rate=3e-4
)
partial_clean = predict_torch(partial_model, transfer_test_loader)
display(partial_history)
display(pd.Series(quality_metrics(partial_clean["y_true"], partial_clean["probs"]), name="partial_resnet18").to_frame())
print(f"Trainable parameters: {partial_trainable:,}; train time: {partial_train_seconds:.2f}s")

## 9. Stress test a deployment shift

`Factory_C` was already a source shift. We now add a capture degradation: lower brightness/contrast, blur, and a downsample-upsample cycle. This is not a universal robustness benchmark; it is a reproducible hypothesis about one plausible operational failure.

In [ ]:
def deployment_shift(image: Image.Image) -> Image.Image:
    shifted = ImageEnhance.Brightness(image).enhance(0.72)
    shifted = ImageEnhance.Contrast(shifted).enhance(0.78)
    shifted = shifted.filter(ImageFilter.GaussianBlur(radius=1.1))
    width, height = shifted.size
    shifted = shifted.resize((width // 2, height // 2), Image.Resampling.BILINEAR)
    return shifted.resize((width, height), Image.Resampling.BILINEAR)


shift_examples = splits["test"].groupby("label", sort=False).head(1)
fig, axes = plt.subplots(len(shift_examples), 2, figsize=(6, 12))
for row, (_, record) in enumerate(shift_examples.iterrows()):
    clean_image = Image.open(record.path).convert("RGB")
    axes[row, 0].imshow(clean_image)
    axes[row, 1].imshow(deployment_shift(clean_image))
    axes[row, 0].set_ylabel(record.label)
    axes[row, 0].set_title("clean")
    axes[row, 1].set_title("shifted")
    axes[row, 0].axis("off")
    axes[row, 1].axis("off")
plt.tight_layout()

In [ ]:
shifted_scratch_loader = DataLoader(
    InspectionDataset(splits["test"], scratch_eval_transform, shift_fn=deployment_shift),
    batch_size=CFG.batch_size,
    shuffle=False,
)
scratch_shift = predict_torch(scratch_model, shifted_scratch_loader)

frozen_shift = {}
for encoder_name, (model, weights, _) in encoders.items():
    shifted_features, shifted_labels, elapsed = extract_features(model, weights, splits["test"], shift_fn=deployment_shift)
    probabilities = probes[encoder_name].predict_proba(shifted_features)
    frozen_shift[encoder_name] = {
        "y_true": shifted_labels,
        "y_pred": probabilities.argmax(axis=1),
        "probs": probabilities,
        "seconds": elapsed,
    }

shifted_partial_loader = DataLoader(
    InspectionDataset(splits["test"], resnet_transform, shift_fn=deployment_shift), batch_size=8, shuffle=False
)
partial_shift = predict_torch(partial_model, shifted_partial_loader)

## 10. Compare models on the evidence that matters

The table includes at least three model strategies and keeps clean and shifted metrics separate. Parameter counts are not all trainable parameters: frozen encoders still carry storage and inference cost. Timings are local measurements for relative context, not deployment benchmarks.

In [ ]:
clean_results = {
    "Scratch CNN": scratch_clean,
    "Frozen ResNet-18 + probe": frozen_clean["resnet18"],
    "Frozen ConvNeXt-Tiny + probe": frozen_clean["convnext_tiny"],
    "Partial ResNet-18": partial_clean,
}
shift_results = {
    "Scratch CNN": scratch_shift,
    "Frozen ResNet-18 + probe": frozen_shift["resnet18"],
    "Frozen ConvNeXt-Tiny + probe": frozen_shift["convnext_tiny"],
    "Partial ResNet-18": partial_shift,
}
model_costs = {
    "Scratch CNN": {"parameters": scratch_parameters, "fit_seconds": scratch_train_seconds},
    "Frozen ResNet-18 + probe": {
        "parameters": sum(p.numel() for p in encoders["resnet18"][0].parameters()),
        "fit_seconds": extraction_seconds["resnet18"],
    },
    "Frozen ConvNeXt-Tiny + probe": {
        "parameters": sum(p.numel() for p in encoders["convnext_tiny"][0].parameters()),
        "fit_seconds": extraction_seconds["convnext_tiny"],
    },
    "Partial ResNet-18": {"parameters": sum(p.numel() for p in partial_model.parameters()), "fit_seconds": partial_train_seconds},
}

comparison_rows = []
for name in clean_results:
    clean_metrics = quality_metrics(clean_results[name]["y_true"], clean_results[name]["probs"])
    shift_metrics = quality_metrics(shift_results[name]["y_true"], shift_results[name]["probs"])
    comparison_rows.append({
        "model": name,
        "clean_accuracy": clean_metrics["accuracy"],
        "clean_macro_f1": clean_metrics["macro_f1"],
        "clean_defect_recall": clean_metrics["defect_recall"],
        "shift_macro_f1": shift_metrics["macro_f1"],
        "shift_defect_recall": shift_metrics["defect_recall"],
        "f1_drop": clean_metrics["macro_f1"] - shift_metrics["macro_f1"],
        "review_rate": clean_metrics["review_rate"],
        "parameters_m": model_costs[name]["parameters"] / 1e6,
        "fit_or_extract_seconds": model_costs[name]["fit_seconds"],
    })
comparison = pd.DataFrame(comparison_rows).sort_values(["shift_defect_recall", "shift_macro_f1"], ascending=False)
display(comparison.style.format({column: "{:.3f}" for column in comparison.columns if column != "model"}))

In [ ]:
source_rows = []
for name, result in clean_results.items():
    for source, positions in splits["test"].groupby("source").groups.items():
        positions = np.asarray(list(positions))
        metrics = quality_metrics(result["y_true"][positions], result["probs"][positions])
        source_rows.append({"model": name, "source": source, **metrics})
display(pd.DataFrame(source_rows))

## 11. Calibration: confidence is not correctness

A model is calibrated when predictions made with confidence $p$ are correct about fraction $p$ of the time. With confidence bins $B_m$, expected calibration error is

$$
\operatorname{ECE}=\sum_{m=1}^{M}\frac{|B_m|}{n}\left|\operatorname{acc}(B_m)-\operatorname{conf}(B_m)\right|.
$$

ECE depends on the bins and sample size, so report the reliability diagram and bin counts too. Here we compare known-source validation with unseen Factory_C plus degradation. We measure miscalibration; we do not pretend this small synthetic set has calibrated the deployment.

In [ ]:
reference_name = "Frozen ResNet-18 + probe"
val_probs = probes["resnet18"].predict_proba(feature_store["resnet18"][boundaries["val"]])
val_true = splits["val"]["label_id"].to_numpy()

def calibration_table(y_true, probabilities, bin_count=10):
    confidence = probabilities.max(axis=1)
    predictions = probabilities.argmax(axis=1)
    correct = predictions == np.asarray(y_true)
    edges = np.linspace(0.0, 1.0, bin_count + 1)
    rows = []
    for index, (left, right) in enumerate(zip(edges[:-1], edges[1:])):
        in_bin = (confidence >= left) & ((confidence < right) if index < bin_count - 1 else (confidence <= right))
        if in_bin.any():
            rows.append({
                "left": left,
                "right": right,
                "count": int(in_bin.sum()),
                "mean_confidence": float(confidence[in_bin].mean()),
                "accuracy": float(correct[in_bin].mean()),
            })
    table = pd.DataFrame(rows)
    table["absolute_gap"] = (table["accuracy"] - table["mean_confidence"]).abs()
    ece = float((table["count"] / len(y_true) * table["absolute_gap"]).sum())
    return table, ece

calibration_sets = {
    "known-source validation": {"y_true": val_true, "probs": val_probs},
    "Factory_C + degradation": frozen_shift["resnet18"],
}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
calibration_rows = []
for name, result in calibration_sets.items():
    table, ece = calibration_table(result["y_true"], result["probs"])
    axes[0].plot(table["mean_confidence"], table["accuracy"], marker="o", label=f"{name} · ECE={ece:.3f}")
    predictions = result["probs"].argmax(axis=1)
    confidence = result["probs"].max(axis=1)
    correct = predictions == result["y_true"]
    axes[1].hist(confidence[correct], bins=np.linspace(0, 1, 11), alpha=0.45, label=f"{name}: correct")
    if (~correct).any():
        axes[1].hist(confidence[~correct], bins=np.linspace(0, 1, 11), histtype="step", linewidth=2, label=f"{name}: wrong")
    calibration_rows.append({
        "slice": name,
        "accuracy": accuracy_score(result["y_true"], predictions),
        "mean_confidence": confidence.mean(),
        "ECE": ece,
    })
axes[0].plot([0, 1], [0, 1], "k--", label="perfect calibration")
axes[0].set(xlabel="mean confidence", ylabel="empirical accuracy", title="Reliability diagram", xlim=(0, 1), ylim=(0, 1))
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.2)
axes[1].set(xlabel="confidence", ylabel="count", title="Confidence for correct and wrong predictions")
axes[1].legend(fontsize=7)
display(pd.DataFrame(calibration_rows))
plt.tight_layout()


## 12. Abstention as an operating policy

A confidence threshold is a business-control choice, not a model metric. The toy cost model below charges 1 unit for review, 25 for an automated missed defect, 3 for an automated false alarm, and 5 for the wrong defect type. It assumes reviewed cases are resolved correctly and instantly—an optimistic assumption that a real pilot must replace with measured reviewer accuracy, capacity, and delay.

In [ ]:
POLICY_COSTS = {"review": 1.0, "false_negative": 25.0, "false_positive": 3.0, "wrong_defect": 5.0}

def policy_metrics(y_true, probabilities, threshold, costs=POLICY_COSTS):
    y_true = np.asarray(y_true)
    prediction = probabilities.argmax(axis=1)
    confidence = probabilities.max(axis=1)
    review = (confidence < threshold) | (prediction == CLASS_TO_ID["Unknown / Ambiguous"])
    automated = ~review
    true_defect = y_true != CLASS_TO_ID["Normal"]
    predicted_defect = prediction != CLASS_TO_ID["Normal"]
    false_negative = automated & true_defect & ~predicted_defect
    false_positive = automated & ~true_defect & predicted_defect
    wrong_defect = automated & true_defect & predicted_defect & (prediction != y_true)
    expected_cost = (
        review.sum() * costs["review"]
        + false_negative.sum() * costs["false_negative"]
        + false_positive.sum() * costs["false_positive"]
        + wrong_defect.sum() * costs["wrong_defect"]
    ) / len(y_true)
    automated_accuracy = float((prediction[automated] == y_true[automated]).mean()) if automated.any() else np.nan
    return {
        "threshold": float(threshold),
        "automation_rate": float(automated.mean()),
        "review_rate": float(review.mean()),
        "automated_accuracy": automated_accuracy,
        "protected_defect_recall": float(1 - false_negative.sum() / max(1, true_defect.sum())),
        "false_negative_rate": float(false_negative.sum() / max(1, true_defect.sum())),
        "false_positive_rate": float(false_positive.sum() / max(1, (~true_defect).sum())),
        "expected_cost_per_case": float(expected_cost),
    }

threshold_grid = np.linspace(0.35, 0.95, 13)
validation_policy = pd.DataFrame([policy_metrics(val_true, val_probs, threshold) for threshold in threshold_grid])
selected_threshold = float(validation_policy.sort_values(["expected_cost_per_case", "review_rate"]).iloc[0].threshold)
display(validation_policy)
print(f"Validation-selected minimum-cost threshold: {selected_threshold:.2f}")

policy_curves = []
for slice_name, result in {
    "known-source validation": {"y_true": val_true, "probs": val_probs},
    "held-out Factory_C": clean_results[reference_name],
    "Factory_C + degradation": shift_results[reference_name],
}.items():
    for threshold in threshold_grid:
        policy_curves.append({"slice": slice_name, **policy_metrics(result["y_true"], result["probs"], threshold)})
policy_curves = pd.DataFrame(policy_curves)
selected_policy = policy_curves[np.isclose(policy_curves["threshold"], selected_threshold)]
display(selected_policy)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for slice_name, group in policy_curves.groupby("slice", sort=False):
    axes[0].plot(group["threshold"], group["automation_rate"], marker="o", label=slice_name)
    axes[1].plot(group["threshold"], group["false_negative_rate"], marker="o", label=slice_name)
    axes[2].plot(group["threshold"], group["expected_cost_per_case"], marker="o", label=slice_name)
for axis, title, ylabel in zip(axes, ["Automation", "Missed-defect risk", "Toy operating cost"], ["automation rate", "false-negative rate", "cost / case"]):
    axis.axvline(selected_threshold, color="black", linestyle="--", alpha=0.6)
    axis.set(xlabel="confidence threshold", ylabel=ylabel, title=title)
    axis.grid(alpha=0.2)
axes[2].legend(fontsize=7)
plt.tight_layout()


## 13. Failure gallery

Aggregate metrics hide operational failure modes. Inspect the highest-confidence errors first: they are the cases least likely to be caught by a simple confidence threshold.

In [ ]:
reference = clean_results[reference_name]
failure_context = "clean"
confidence = reference["probs"].max(axis=1)
wrong = reference["y_pred"] != reference["y_true"]
if not wrong.any():
    reference = shift_results[reference_name]
    failure_context = "shifted"
    confidence = reference["probs"].max(axis=1)
    wrong = reference["y_pred"] != reference["y_true"]
failure_order = np.argsort(np.where(wrong, confidence, -1))[::-1]
failure_indices = [index for index in failure_order if wrong[index]][: min(8, int(wrong.sum()))]
if failure_indices:
    columns = 4
    rows = int(np.ceil(len(failure_indices) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(12, 3 * rows), squeeze=False)
    for axis in axes.flat:
        axis.axis("off")
    for axis, index in zip(axes.flat, failure_indices):
        record = splits["test"].iloc[index]
        failure_image = Image.open(record.path).convert("RGB")
        if failure_context == "shifted":
            failure_image = deployment_shift(failure_image)
        axis.imshow(failure_image)
        axis.set_title(
            f"true: {CLASS_NAMES[reference['y_true'][index]]}\npred: {CLASS_NAMES[reference['y_pred'][index]]}\nconf: {confidence[index]:.2f}",
            fontsize=8,
        )
        axis.axis("off")
    fig.suptitle(f"Highest-confidence {failure_context} held-out failures", y=1.02)
    plt.tight_layout()
else:
    print("No clean or shifted errors; add a harder source slice instead of declaring the system solved.")

In [ ]:
reference = clean_results[reference_name]
policy_summary = {
    "slice": "held-out Factory_C",
    **policy_metrics(reference["y_true"], reference["probs"], selected_threshold),
}
display(pd.Series(policy_summary, name="held_out_policy").to_frame())


### Grad-CAM for correct and incorrect predictions

Grad-CAM weights the last convolutional feature maps by the gradient of a selected class score. It can suggest whether the scratch CNN attended to the crack, plate edge, background, or another shortcut. It is a qualitative diagnostic—not a causal explanation or proof of correctness.

In [ ]:
def grad_cam(model, input_tensor, target_class=None):
    captured = {}

    def save_activation(_module, _inputs, output):
        captured["activation"] = output
        output.register_hook(lambda gradient: captured.update(gradient=gradient))

    handle = model.features[8].register_forward_hook(save_activation)
    try:
        model.zero_grad(set_to_none=True)
        logits = model(input_tensor)
        target = int(logits.argmax(dim=1).item()) if target_class is None else int(target_class)
        logits[0, target].backward()
        activation = captured["activation"].detach()[0]
        gradient = captured["gradient"].detach()[0]
        weights = gradient.mean(dim=(1, 2), keepdim=True)
        heatmap = F.relu((weights * activation).sum(dim=0))[None, None]
        heatmap = F.interpolate(heatmap, size=input_tensor.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        heatmap = heatmap / (heatmap.max() + 1e-8)
        return heatmap.cpu().numpy(), target
    finally:
        handle.remove()

cases = []
clean_correct = np.flatnonzero(scratch_clean["y_pred"] == scratch_clean["y_true"])
if len(clean_correct):
    cases.append(("correct / clean", int(clean_correct[0]), False, scratch_clean))
clean_wrong = np.flatnonzero(scratch_clean["y_pred"] != scratch_clean["y_true"])
if len(clean_wrong):
    cases.append(("incorrect / clean", int(clean_wrong[np.argmax(scratch_clean["probs"][clean_wrong].max(axis=1))]), False, scratch_clean))
else:
    shift_wrong = np.flatnonzero(scratch_shift["y_pred"] != scratch_shift["y_true"])
    if len(shift_wrong):
        cases.append(("incorrect / shifted", int(shift_wrong[np.argmax(scratch_shift["probs"][shift_wrong].max(axis=1))]), True, scratch_shift))

if cases:
    fig, axes = plt.subplots(len(cases), 2, figsize=(7, 3.2 * len(cases)), squeeze=False)
    for row_number, (case_name, index, shifted, result) in enumerate(cases):
        record = splits["test"].iloc[index]
        image = Image.open(record.path).convert("RGB")
        if shifted:
            image = deployment_shift(image)
        input_tensor = scratch_eval_transform(image).unsqueeze(0).to(DEVICE).requires_grad_(True)
        heatmap, predicted_class = grad_cam(scratch_model, input_tensor)
        resized_image = np.asarray(image.resize((CFG.image_size, CFG.image_size)), dtype=np.float32) / 255.0
        coloured_heatmap = plt.cm.inferno(heatmap)[..., :3]
        overlay = np.clip(0.55 * resized_image + 0.45 * coloured_heatmap, 0, 1)
        axes[row_number, 0].imshow(resized_image)
        axes[row_number, 1].imshow(overlay)
        axes[row_number, 0].set_title(f"{case_name}\ntrue: {record.label}", fontsize=9)
        axes[row_number, 1].set_title(f"Grad-CAM\npred: {CLASS_NAMES[predicted_class]}", fontsize=9)
        axes[row_number, 0].axis("off")
        axes[row_number, 1].axis("off")
    plt.tight_layout()
else:
    print("No correct/error pair is available in this tiny run; increase CV_FULL_RUN or add a harder slice.")


## 14. Enterprise decision and saved evidence

In [ ]:
candidate = comparison.iloc[0]
decision = {
    "course": "Modern Computer Vision Foundations",
    "scenario": "five-class enterprise visual quality inspection",
    "data_scope": "deterministic synthetic images; Factory_C held out",
    "candidate_model": candidate["model"],
    "recommendation": "limited assistive pilot with mandatory human review; no autonomous deployment",
    "evidence": {
        "clean_macro_f1": round(float(candidate["clean_macro_f1"]), 4),
        "shift_macro_f1": round(float(candidate["shift_macro_f1"]), 4),
        "clean_defect_recall": round(float(candidate["clean_defect_recall"]), 4),
        "shift_defect_recall": round(float(candidate["shift_defect_recall"]), 4),
        "f1_drop": round(float(candidate["f1_drop"]), 4),
        "reference_policy": policy_summary,
    },
    "risk_boundaries": [
        "Synthetic data does not establish real defect prevalence or capture diversity.",
        "The stress test covers one constructed degradation, not the deployment distribution.",
        "Calibration and policy costs were measured only on a small synthetic validation set; scores were not recalibrated.",
        "Unknown/Ambiguous and low-confidence cases require human review.",
    ],
    "required_next_steps": [
        "Acquire licensed representative data with source, time, and product lineage.",
        "Define defect-miss costs and reviewer capacity with domain owners.",
        "Repeat grouped and temporal validation across facilities and devices.",
        "Recalibrate scores and validate review costs/capacity on representative real outcomes.",
        "Run shadow mode with outcome monitoring, rollback triggers, and audit logs.",
    ],
    "monitor": [
        "input validity and source mix",
        "embedding and confidence drift",
        "class, abstention, and human-override rates",
        "per-source defect recall and delayed outcomes",
        "latency, errors, and model/preprocessing versions",
    ],
    "configuration": asdict(CFG),
    "versions": versions.to_dict(),
}
decision_path = ARTIFACT_DIR / "enterprise_decision.json"
decision_path.write_text(json.dumps(decision, indent=2), encoding="utf-8")
print(json.dumps(decision, indent=2))
print("Saved:", decision_path)

## 15. Optional real-world extension: VisA

The default lab stays credential-free and deterministic. To repeat the workflow on a licensed industrial anomaly dataset, download the **VisA** dataset from the [AWS Registry of Open Data](https://registry.opendata.aws/visa/) or follow the [official Amazon Science repository](https://github.com/amazon-science/spot-diff), then set `CV_VISA_DIR` to the extracted dataset root.

The guarded cell below uses the same common libraries—Pillow, pandas, scikit-learn, PyTorch, and torchvision—to profile the files, create a bounded exploratory split, fit a frozen ResNet-18 probe, and inspect nearest-neighbour label disagreements. A random exploratory split is not deployment evidence; replace it with product/source/time groups for a real study.

In [ ]:
visa_environment = os.getenv("CV_VISA_DIR")
if not visa_environment:
    print("Optional VisA extension skipped. Set CV_VISA_DIR to an extracted VisA root to run it.")
else:
    visa_root = Path(visa_environment).expanduser().resolve()
    image_paths = sorted(path for path in visa_root.rglob("*") if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})

    def visa_record(path):
        relative_parts = path.relative_to(visa_root).parts
        lower_parts = [part.lower() for part in relative_parts]
        normal = any(part == "normal" or part == "good" for part in lower_parts)
        anomalous = any(part in {"anomaly", "anomalous", "bad"} for part in lower_parts)
        if not normal and not anomalous:
            return None
        marker_index = next((index for index, part in enumerate(lower_parts) if part in {"normal", "good", "anomaly", "anomalous", "bad"}), len(lower_parts) - 1)
        source = relative_parts[max(0, marker_index - 1)]
        with Image.open(path) as image:
            width, height = image.size
        return {"path": str(path), "label": "normal" if normal else "anomaly", "label_id": 0 if normal else 1, "source": source, "width": width, "height": height}

    records = [record for path in image_paths if (record := visa_record(path)) is not None]
    visa_frame = pd.DataFrame(records)
    if visa_frame.empty or visa_frame["label_id"].nunique() < 2:
        print("No recognizable Normal/Anomaly directory layout found. Point CV_VISA_DIR at the extracted image tree.")
    else:
        visa_frame = (
            visa_frame.groupby("label", group_keys=False)
            .apply(lambda group: group.sample(min(len(group), 250), random_state=SEED), include_groups=False)
            .reset_index(drop=True)
        )
        visa_frame["label"] = visa_frame["label_id"].map({0: "normal", 1: "anomaly"})
        display(visa_frame.groupby(["source", "label"])[["width", "height"]].agg(["count", "min", "max"]))
        visa_train, visa_holdout = train_test_split(visa_frame, test_size=0.40, stratify=visa_frame["label_id"], random_state=SEED)
        visa_val, visa_test = train_test_split(visa_holdout, test_size=0.50, stratify=visa_holdout["label_id"], random_state=SEED)
        visa_encoder, visa_weights, _ = encoders["resnet18"]
        visa_train_features, visa_train_labels, _ = extract_features(visa_encoder, visa_weights, visa_train.reset_index(drop=True))
        visa_test_features, visa_test_labels, _ = extract_features(visa_encoder, visa_weights, visa_test.reset_index(drop=True))
        visa_probe = LogisticRegression(max_iter=600, class_weight="balanced", random_state=SEED).fit(visa_train_features, visa_train_labels)
        visa_probs = visa_probe.predict_proba(visa_test_features)
        display(pd.Series({
            "test_images": len(visa_test),
            "accuracy": accuracy_score(visa_test_labels, visa_probs.argmax(axis=1)),
            "macro_f1": f1_score(visa_test_labels, visa_probs.argmax(axis=1), average="macro", zero_division=0),
        }, name="VisA exploratory probe").to_frame())
        visa_similarity = normalize(visa_test_features) @ normalize(visa_train_features).T
        visa_nearest = visa_similarity.argmax(axis=1)
        disagreements = visa_train_labels[visa_nearest] != visa_test_labels
        print(f"Nearest-neighbour label disagreements: {int(disagreements.sum())}/{len(disagreements)}")


## 16. Interpret, extend, and transition forward

Before calling the lab complete, answer these evidence questions:

1. Which tensor mistake was visible in both statistics and pixels, and which might evade a quick visual check?
2. Why did the manual cross-correlation and `F.conv2d` agree, and how do the learned filters differ from Sobel?
3. Where did empirical receptive-field support shrink below the theoretical maximum?
4. Did nearest-neighbour review reveal source shortcuts, hard examples, or label disagreement?
5. How did reliability and operating cost change under Factory_C degradation?
6. Did Grad-CAM focus on the defect, the plate geometry, or a background cue—and what evidence would test that hypothesis?
7. Which assumptions prevent this synthetic experiment from authorizing autonomous inspection?

### What you should now be able to explain without code

Answer each in two or three sentences:

1. Why might 97% validation accuracy still produce an unsafe inspection system?
2. Why can a pretrained encoder outperform a CNN trained specifically for your dataset?
3. What is the difference between a feature map and an embedding?
4. Why can confidence increase while reliability decreases?
5. Why must train/test splitting sometimes happen by factory rather than by image?
6. What does a receptive field tell us—and what does it not tell us?
7. Why is the highest-accuracy model not automatically the best deployment choice?
8. What changes as we move from task-specific CNNs toward foundation vision models?

If an answer depends only on a metric value or library name, revisit the corresponding experiment.

Next: **Course 02 — Modern CNN Architectures & Efficient Vision**, where architecture choice becomes a controlled quality/latency/memory benchmark. Later beginner courses separately cover transformers, self-supervision, detection, promptable segmentation, embeddings/retrieval, tracking/pose, and foundation/open-vocabulary vision.